In [ ]:
import pandas as pd
import regex as re

In [ ]:
df_radars = pd.read_csv('radars.csv')
df_drivers = pd.read_csv('drivers.csv')
df_fines = pd.read_csv('fines.csv')

In [ ]:
df_radars.head()

Ejercicio 1 

In [ ]:
def unique_radars(df_radars:pd.DataFrame):
    return df_radars['id'].drop_duplicates().count()
unique_radars(df_radars)

In [ ]:
def unique_per_type(df_radars:pd.DataFrame):
    df = df_radars.drop_duplicates(subset = 'id').copy()
    df = df.replace({'Pegass':'Pegasus',' Velocidad':'Velocidad','tramo':'Tramo'})
    return df.groupby('type')['id'].count()

unique_per_type(df_radars)

In [ ]:
def remove_duplicates(df_radars:pd.DataFrame):
    return df_radars.drop_duplicates()
df = remove_duplicates(df_radars)
df.head()

Ejercicio 2

In [ ]:
import numpy as np
dict_geografia = {'S':36,'N':43.794,'E':3.320,'W':9.9296}
def spanish_radars(df_radars:pd.DataFrame):
    "42º43'38.16"" N, 1º7'36.42"" W"
    patron = re.compile(r'(\d{1,2})\º(\d{1,2})\'(\d{1,2}\.\d{1,2})\"+\s+(\w)\,\s+(\d{1,2})\º(\d{1,2})\'(\d{1,2}\.\d{1,2})\"+\s+(\w)')
    def cambio(celda):
        if celda is not None:
            m = patron.search(celda)
            grados_lat = int(m.group(1))
            minutos_lat = int(m.group(2))
            segundos_lat = float(m.group(3))
            sentido_lat = m.group(4)
            total_grados_lat = grados_lat+minutos_lat/60+segundos_lat/3600
            
            grados_long = int(m.group(5))
            minutos_long = int(m.group(6))
            segundos_long = float(m.group(7))
            sentido_long = m.group(8)
            total_grados_long = grados_long+minutos_long/60+segundos_long/3600
            
            if total_grados_lat <= dict_geografia[sentido_lat] and total_grados_long <= dict_geografia[sentido_long]:
                return f'{total_grados_lat} {sentido_lat}, {total_grados_long} {sentido_long}'
            else:
                return np.nan
        else:
            return np.nan
    df_radars['location']= df_radars['location'].apply(cambio)
    return df_radars.dropna().copy()
spanish_radars(df_radars)


In [ ]:
def plot_boxplot(df_fines:pd.DataFrame):
    return df_fines['speed'].plot(kind='box')

plot_boxplot(df_fines)

In [ ]:
def asses_speed_camera(df:pd.DataFrame):
    df['z_score'] = abs((df['speed']-df['speed'].mean())/df['speed'].std())
    return list(df.loc[df['z_score']>3,'radar_id'].unique())

asses_speed_camera(df_fines)

In [ ]:
def points_per_driver(df_drivers:pd.DataFrame,df_fines:pd.DataFrame,df_radars:pd.DataFrame):
    df_fines=df_fines.rename(columns={'points':'minus_points'})
    df = df_drivers.merge(df_fines, on='driver_id')
    df_new = pd.DataFrame()
    df_new['driver_id'] = df['driver_id']
    df_new['remaining points'] = df['points']-df['minus_points']
    return df_new

points_per_driver(df_drivers,df_fines,df_radars)

In [ ]:
def new_speed_limit(df_fines:pd.DataFrame):
    velocidad_media = df_fines['speed'].sum()/df_fines['speed'].count()
    multa_media = df_fines['fine'].sum()/df_fines['fine'].count()
    velocidad_adecuada = velocidad_media*multa_media/(multa_media/2)
    return velocidad_adecuada-velocidad_media
new_speed_limit(df_fines)